# 语言模型零基础 01：从计数到 Bigram

这是一份专门为 **OpenFst / N-gram 零基础学习路线**准备的入口 Notebook。你不需要先学完前 18 课。

完成本课后，你应该能够：

1. 说清语言模型和声学模型各自负责什么；
2. 从小语料中数出 unigram 和 bigram；
3. 计算条件概率与一个句子的 Bigram 概率；
4. 亲眼看到“未见搭配导致整句概率为 0”；
5. 用 Add-k 平滑消除零概率；
6. 理解为什么 WFST 使用 `-log(probability)` 作为路径代价。


## 使用方法

按顺序逐格运行。看到 **先猜再运行** 时，先在心里或 Markdown cell 写下预测。运行后只改一个变量，再观察结果。

> 本课用空格分词，例如 `我 爱 学习`。`<s>` 表示句首，`</s>` 表示句尾，它们不是普通词。


## 1. ASR 为什么需要语言模型？

ASR 常被概括为寻找：

$$W^* = \arg\max_W P(X|W)P(W)$$

- `P(X|W)`：这句话读出来是否像听到的声音——声学模型；
- `P(W)`：这串词本身是否自然——语言模型。

声音含糊时，语言模型会让“今天天气很好”通常优先于“今天田七很好”。本课只研究 `P(W)`。


In [1]:
from collections import Counter
from math import isclose, log, prod

corpus = [
    "我 爱 你",
    "我 爱 学习",
    "你 爱 学习",
    "我 学习 ASR",
]

def tokens(sentence):
    return sentence.strip().split()

def with_boundaries(sentence):
    return ["<s>", *tokens(sentence), "</s>"]

for sentence in corpus:
    print(with_boundaries(sentence))


['<s>', '我', '爱', '你', '</s>']
['<s>', '我', '爱', '学习', '</s>']
['<s>', '你', '爱', '学习', '</s>']
['<s>', '我', '学习', 'ASR', '</s>']


## 2. 从语料中数 Bigram

Bigram 假设：预测下一个词时，只看前一个词。

条件概率的最大似然估计：

$$P(b|a)=\frac{C(a,b)}{C(a)}$$

直白地说：`a` 后面出现 `b` 的次数，除以 `a` 后面出现过词的总次数。


In [2]:
def count_bigrams(sentences):
    context_counts = Counter()
    bigram_counts = Counter()
    vocabulary = set()
    for sentence in sentences:
        seq = with_boundaries(sentence)
        vocabulary.update(seq[1:])  # 可作为“下一个词”的词，包括 </s>
        for previous, current in zip(seq, seq[1:]):
            context_counts[previous] += 1
            bigram_counts[previous, current] += 1
    return context_counts, bigram_counts, sorted(vocabulary)

context_counts, bigram_counts, vocabulary = count_bigrams(corpus)

print("“我”后面共出现过：", context_counts["我"], "次")
for (previous, current), count in sorted(bigram_counts.items()):
    if previous == "我":
        print(f"C({previous}, {current}) = {count}")


“我”后面共出现过： 3 次
C(我, 学习) = 1
C(我, 爱) = 2


In [3]:
def p_bigram(current, previous, k=0.0):
    """Add-k Bigram 概率；k=0 时就是未经平滑的简单计数。"""
    V = len(vocabulary)
    denominator = context_counts[previous] + k * V
    if denominator == 0:
        return 0.0
    return (bigram_counts[previous, current] + k) / denominator

print(f"P(爱 | 我)   = {p_bigram('爱', '我'):.3f}")
print(f"P(学习 | 我) = {p_bigram('学习', '我'):.3f}")
print("两者相加 =", p_bigram('爱', '我') + p_bigram('学习', '我'))


P(爱 | 我)   = 0.667
P(学习 | 我) = 0.333
两者相加 = 1.0


### 你的第一个可修改实验

把下面的 `previous` 改成 `爱`、`学习` 或 `<s>`，看看它后面各个词的概率。然后回到语料中手工核对一次。


In [4]:
previous = "爱"  # 请修改这里
distribution = [(word, p_bigram(word, previous)) for word in vocabulary]
distribution = [(word, p) for word, p in distribution if p > 0]
for word, probability in sorted(distribution, key=lambda item: -item[1]):
    print(f"P({word} | {previous}) = {probability:.3f}")
print("概率之和 =", sum(p for _, p in distribution))


P(学习 | 爱) = 0.667
P(你 | 爱) = 0.333
概率之和 = 1.0


## 3. 计算整个句子的概率

Bigram 模型把每一步条件概率相乘：

$$P(w_1,\dots,w_n) \approx P(w_1|<s>)P(w_2|w_1)\cdots P(</s>|w_n)$$


In [5]:
def sentence_probability(sentence, k=0.0, verbose=True):
    seq = with_boundaries(sentence)
    steps = []
    for previous, current in zip(seq, seq[1:]):
        probability = p_bigram(current, previous, k=k)
        steps.append((previous, current, probability))
    if verbose:
        for previous, current, probability in steps:
            print(f"P({current} | {previous}) = {probability:.6f}")
        print("整句概率 =", prod(p for _, _, p in steps))
    return prod(p for _, _, p in steps)

sentence_probability("我 爱 学习")


P(我 | <s>) = 0.750000
P(爱 | 我) = 0.666667
P(学习 | 爱) = 0.666667
P(</s> | 学习) = 0.666667
整句概率 = 0.2222222222222222


0.2222222222222222

## 4. 零概率问题与 Add-k 平滑

**先猜再运行：** `你 学习 ASR` 很合理，但训练语料没有出现 `你 学习`。未经平滑时，整句概率会是多少？

Add-k 的公式是：

$$P_k(b|a)=\frac{C(a,b)+k}{C(a)+kV}$$

它会从出现过的搭配中拿出一点概率，分给没出现过的搭配。它适合教学，但大型实用系统通常使用更好的 Kneser–Ney 等方法。


In [6]:
print("不平滑：")
sentence_probability("你 学习 ASR", k=0.0)
print("\nAdd-0.1 平滑：")
sentence_probability("你 学习 ASR", k=0.1)


不平滑：
P(你 | <s>) = 0.250000
P(学习 | 你) = 0.000000
P(ASR | 学习) = 0.333333
P(</s> | ASR) = 1.000000
整句概率 = 0.0

Add-0.1 平滑：
P(你 | <s>) = 0.239130
P(学习 | 你) = 0.038462
P(ASR | 学习) = 0.305556
P(</s> | ASR) = 0.687500
整句概率 = 0.001932076830174657


0.001932076830174657

### 滑块实验：平滑强度怎样改变概率？

拖动 `k`，同时观察未见搭配 `P(学习|你)` 和已见搭配 `P(爱|你)`。`k` 不是越大越好。


In [7]:
import ipywidgets as widgets
from IPython.display import display

def smoothing_demo(k=0.1):
    unseen = p_bigram("学习", "你", k=k)
    seen = p_bigram("爱", "你", k=k)
    print(f"k = {k:.2f}")
    print(f"未见搭配 P(学习 | 你) = {unseen:.6f}")
    print(f"已见搭配 P(爱 | 你)   = {seen:.6f}")

widgets.interact(smoothing_demo, k=widgets.FloatSlider(
    value=0.1, min=0.0, max=2.0, step=0.1, description="k"
));


interactive(children=(FloatSlider(value=0.1, description='k', max=2.0), Output()), _dom_classes=('widget-inter…

## 5. 从概率走向 OpenFst：负对数代价

概率路径要相乘；图算法更方便累加代价。因此定义：

$$cost=-\log(P)$$

概率越大，代价越小；整条路径概率最大，等价于整条路径代价最小。这就是以后 OpenFst `shortestpath` 的概率直觉。


In [8]:
sentence = "我 爱 学习"
seq = with_boundaries(sentence)
probabilities = [p_bigram(current, previous) for previous, current in zip(seq, seq[1:])]
costs = [-log(p) for p in probabilities]

print("每一步概率：", [round(p, 4) for p in probabilities])
print("概率相乘：  ", prod(probabilities))
print("每一步代价：", [round(c, 4) for c in costs])
print("代价相加：  ", sum(costs))
print("exp(-总代价) 与整句概率一致：", __import__('math').exp(-sum(costs)))


每一步概率： [0.75, 0.6667, 0.6667, 0.6667]
概率相乘：   0.2222222222222222
每一步代价： [0.2877, 0.4055, 0.4055, 0.4055]
代价相加：   1.5040773967762742
exp(-总代价) 与整句概率一致： 0.2222222222222222


## 6. 自动判题

先自己计算，再填写三个答案。全部通过后再进入下一课。


In [9]:
# 请修改下面三个值
answer_1 = 1 / 3  # P(你 | 爱)
answer_2 = 1 / 3  # P(ASR | 学习)
answer_3 = 1 / 12 # P(我 学习 ASR)

expected = [1/3, 1/3, 1/12]
answers = [answer_1, answer_2, answer_3]
labels = ["P(你 | 爱)", "P(ASR | 学习)", "P(我 学习 ASR)"]

passed = 0
for label, answer, target in zip(labels, answers, expected):
    ok = isclose(answer, target, rel_tol=1e-9, abs_tol=1e-9)
    passed += int(ok)
    print(("✅" if ok else "❌"), label, "你的答案：", answer)
print(f"\n得分：{passed}/3")
if passed == 3:
    print("很好：你已经掌握最基础的 Bigram 计数，可以进入回退与插值。")
else:
    print("回到第 2～3 节，逐项写出分子和分母，再试一次。")


✅ P(你 | 爱) 你的答案： 0.3333333333333333
✅ P(ASR | 学习) 你的答案： 0.3333333333333333
✅ P(我 学习 ASR) 你的答案： 0.08333333333333333

得分：3/3
很好：你已经掌握最基础的 Bigram 计数，可以进入回退与插值。


## 离场票

不看上文，尝试口头回答：

1. `P(学习|爱)` 的分子和分母分别是什么？
2. 为什么一个 Bigram 概率为 0 会让整句概率为 0？
3. 平滑解决什么问题？
4. 为什么概率最大路径等价于 `-log` 代价最小路径？

下一步建议：完成本课自动判题后学习 `语言模型零基础_02_平滑回退OOV与困惑度.ipynb`，然后再进入 FSA/FST 与真实 OpenFst 实验。
